In [2]:
import pandas as pa
import numpy as np

#  1. Data Layer: Real-Time Price and News Fetcher

In [ ]:
# data_fetchers.py
import requests
import yfinance as yf
import os
from datetime import datetime
from typing import Dict, List, Optional
from dataclasses import dataclass

@dataclass
class PriceData:
    symbol: str
    price: float
    timestamp: datetime

@dataclass
class NewsArticle:
    title: str
    summary: str
    url: str
    published_at: datetime
    source: str

class GoldSilverPriceFetcher:
    """Fetches real-time gold and silver prices from multiple free sources."""
    
    def __init__(self):
        self.gold_symbol = "GC=F"      # Gold futures on Yahoo Finance
        self.silver_symbol = "SI=F"    # Silver futures on Yahoo Finance
        
    def get_current_prices(self) -> Dict[str, PriceData]:
        """Get current gold and silver prices."""
        result = {}
        
        try:
            gold = yf.Ticker(self.gold_symbol)
            gold_info = gold.history(period="1d", interval="1m")
            if not gold_info.empty:
                gold_price = gold_info['Close'].iloc[-1]
                result["GOLD"] = PriceData("GOLD", gold_price, datetime.now())
                
            silver = yf.Ticker(self.silver_symbol)
            silver_info = silver.history(period="1d", interval="1m")
            if not silver_info.empty:
                silver_price = silver_info['Close'].iloc[-1]
                result["SILVER"] = PriceData("SILVER", silver_price, datetime.now())
                
        except Exception as e:
            print(f"Error fetching prices: {e}")
            
        return result
    
    def get_historical_prices(self, symbol: str, period: str = "6mo") -> pd.DataFrame:
        """Get historical OHLCV data for backtesting."""
        ticker = yf.Ticker(symbol)
        return ticker.history(period=period)

class NewsFetcher:
    """Fetches gold and silver related news articles."""
    
    def __init__(self):
        self.finnhub_api_key = os.getenv("FINNHUB_API_KEY")
        self.tavily_api_key = os.getenv("TAVILY_API_KEY")
        
    def fetch_gold_silver_news_finnhub(self, days_back: int = 7) -> List[NewsArticle]:
        """Fetch news using Finnhub API (free tier: 60 calls/min)."""
        articles = []
        
        if not self.finnhub_api_key:
            return articles
            
        url = f"https://finnhub.io/api/v1/news?category=general&token={self.finnhub_api_key}"
        
        try:
            response = requests.get(url)
            if response.status_code == 200:
                news_data = response.json()
                for item in news_data[:20]:  # Limit to 20 most recent
                    title = item.get('headline', '')
                    if any(keyword in title.lower() for keyword in ['gold', 'silver', 'precious', 'metal']):
                        articles.append(NewsArticle(
                            title=title,
                            summary=item.get('summary', ''),
                            url=item.get('url', ''),
                            published_at=datetime.fromtimestamp(item.get('datetime', 0)),
                            source='Finnhub'
                        ))
        except Exception as e:
            print(f"Error fetching news: {e}")
            
        return articles
    
    def search_tavily(self, query: str, max_results: int = 5) -> List[Dict]:
        """Search for news using Tavily API (free tier available)."""
        if not self.tavily_api_key:
            return []
            
        try:
            from tavily import TavilyClient
            client = TavilyClient(api_key=self.tavily_api_key)
            response = client.search(query=query, max_results=max_results)
            return response.get('results', [])
        except ImportError:
            print("Tavily package not installed. Run: pip install tavily-python")
            return []

# 2. The Multi-Agent System with LangGraph
Now let's build the core brain — a LangGraph-based multi-agent system with specialized agents for technical analysis, sentiment analysis, and risk management.

In [1]:
# trading_agent.py
import os
from typing import TypedDict, Annotated, List, Dict, Any
from datetime import datetime
import operator
import pandas as pd
import numpy as np

from langgraph.graph import StateGraph, END
from langgraph.checkpoint import MemorySaver
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool

from data_fetchers import GoldSilverPriceFetcher, NewsFetcher

# Initialize components
price_fetcher = GoldSilverPriceFetcher()
news_fetcher = NewsFetcher()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Define the state that flows through our agent graph
class TradingState(TypedDict):
    messages: Annotated[List, operator.add]
    gold_price: float
    silver_price: float
    technical_signals: Dict[str, Any]
    sentiment_score: float
    sentiment_details: List[str]
    risk_assessment: Dict[str, Any]
    final_decision: Dict[str, Any]
    timestamp: str

# ----- Technical Analysis Tools -----
@tool
def calculate_rsi(prices: List[float], period: int = 14) -> float:
    """Calculate RSI indicator for gold or silver."""
    if len(prices) < period + 1:
        return 50.0
    
    deltas = [prices[i] - prices[i-1] for i in range(1, len(prices))]
    gains = [d if d > 0 else 0 for d in deltas]
    losses = [-d if d < 0 else 0 for d in deltas]
    
    avg_gain = sum(gains[-period:]) / period
    avg_loss = sum(losses[-period:]) / period
    
    if avg_loss == 0:
        return 100.0
    
    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

@tool
def calculate_macd(prices: List[float]) -> Dict[str, float]:
    """Calculate MACD (Moving Average Convergence Divergence)."""
    if len(prices) < 26:
        return {"macd": 0, "signal": 0, "histogram": 0}
    
    ema12 = pd.Series(prices).ewm(span=12, adjust=False).mean().iloc[-1]
    ema26 = pd.Series(prices).ewm(span=26, adjust=False).mean().iloc[-1]
    macd = ema12 - ema26
    signal = pd.Series([macd]).ewm(span=9, adjust=False).mean().iloc[-1]
    
    return {"macd": macd, "signal": signal, "histogram": macd - signal}

@tool
def calculate_bollinger_bands(prices: List[float], period: int = 20, std_dev: float = 2) -> Dict[str, float]:
    """Calculate Bollinger Bands."""
    if len(prices) < period:
        return {"upper": prices[-1], "middle": prices[-1], "lower": prices[-1]}
    
    series = pd.Series(prices[-period:])
    middle = series.mean()
    std = series.std()
    upper = middle + (std * std_dev)
    lower = middle - (std * std_dev)
    
    return {"upper": upper, "middle": middle, "lower": lower}

# ----- Agent Nodes -----
def technical_analyst_node(state: TradingState) -> TradingState:
    """Agent 1: Analyzes technical indicators for gold and silver."""
    
    # Fetch historical price data for gold
    gold_data = price_fetcher.get_historical_prices("GC=F", period="3mo")
    gold_prices = gold_data['Close'].tolist() if not gold_data.empty else []
    
    if len(gold_prices) > 0:
        rsi = calculate_rsi.invoke({"prices": gold_prices})
        macd = calculate_macd.invoke({"prices": gold_prices})
        bb = calculate_bollinger_bands.invoke({"prices": gold_prices})
        
        # Generate signal based on indicators
        signal = "NEUTRAL"
        confidence = 0.5
        
        if rsi < 30 and macd['macd'] > macd['signal']:
            signal = "BUY"
            confidence = 0.75
        elif rsi > 70 and macd['macd'] < macd['signal']:
            signal = "SELL"
            confidence = 0.75
        elif rsi < 25:
            signal = "BUY"
            confidence = 0.6
        elif rsi > 75:
            signal = "SELL"
            confidence = 0.6
            
        technical_signals = {
            "gold": {
                "signal": signal,
                "confidence": confidence,
                "rsi": rsi,
                "macd": macd,
                "bollinger_bands": bb
            }
        }
        
        # Analyze silver similarly
        silver_data = price_fetcher.get_historical_prices("SI=F", period="3mo")
        silver_prices = silver_data['Close'].tolist() if not silver_data.empty else []
        if silver_prices:
            silver_rsi = calculate_rsi.invoke({"prices": silver_prices})
            silver_signal = "NEUTRAL"
            if silver_rsi < 30:
                silver_signal = "BUY"
            elif silver_rsi > 70:
                silver_signal = "SELL"
            technical_signals["silver"] = {
                "signal": silver_signal,
                "confidence": 0.6 if silver_signal != "NEUTRAL" else 0.4,
                "rsi": silver_rsi
            }
    else:
        technical_signals = {"gold": {"signal": "NEUTRAL", "confidence": 0.5}}
    
    state["technical_signals"] = technical_signals
    return state

def sentiment_analyst_node(state: TradingState) -> TradingState:
    """Agent 2: Analyzes sentiment from news headlines."""
    
    news_articles = news_fetcher.fetch_gold_silver_news_finnhub(days_back=3)
    
    if not news_articles:
        state["sentiment_score"] = 0.5
        state["sentiment_details"] = ["No news available for analysis"]
        return state
    
    # Prepare news text for LLM analysis
    news_texts = [f"- {article.title}: {article.summary[:200]}" for article in news_articles[:10]]
    news_prompt = f"""
    Analyze the following gold and silver related news headlines and determine the overall market sentiment.
    
    News:
    {chr(10).join(news_texts)}
    
    Respond with a JSON object containing:
    - sentiment_score: A number between 0 and 1 (0 = extremely bearish, 0.5 = neutral, 1 = extremely bullish)
    - key_drivers: List of 2-3 main factors driving the sentiment
    - confidence: Your confidence in this assessment (0-1)
    """
    
    try:
        response = llm.invoke([SystemMessage(content="You are a financial sentiment analyst."),
                              HumanMessage(content=news_prompt)])
        
        # Parse the response (simplified extraction)
        sentiment_score = 0.5
        key_drivers = []
        
        if "sentiment_score" in response.content.lower():
            # Simple extraction; in production use structured output
            sentiment_score = 0.6  # default slightly bullish
        
        if "bullish" in response.content.lower():
            sentiment_score = 0.7
        elif "bearish" in response.content.lower():
            sentiment_score = 0.3
            
        key_drivers = [d.strip() for d in response.content.split('\n') if '-' in d][:3]
        
    except Exception as e:
        print(f"Sentiment analysis error: {e}")
        sentiment_score = 0.5
        key_drivers = ["Analysis unavailable"]
    
    state["sentiment_score"] = sentiment_score
    state["sentiment_details"] = key_drivers
    return state

def risk_manager_node(state: TradingState) -> TradingState:
    """Agent 3: Assesses market risk conditions."""
    
    # Calculate ATR for stop-loss placement
    gold_data = price_fetcher.get_historical_prices("GC=F", period="20d")
    
    if not gold_data.empty:
        # Calculate ATR (Average True Range)
        high_low = gold_data['High'] - gold_data['Low']
        high_close = abs(gold_data['High'] - gold_data['Close'].shift())
        low_close = abs(gold_data['Low'] - gold_data['Close'].shift())
        tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
        atr = tr.rolling(window=14).mean().iloc[-1]
        
        # Calculate current volatility (standard deviation of returns)
        returns = gold_data['Close'].pct_change().dropna()
        volatility = returns.std() * np.sqrt(252)  # Annualized
        
        # Determine risk level
        risk_level = "MODERATE"
        if volatility > 0.25:
            risk_level = "HIGH"
        elif volatility < 0.15:
            risk_level = "LOW"
            
        state["risk_assessment"] = {
            "risk_level": risk_level,
            "atr": atr,
            "volatility": volatility,
            "position_size_multiplier": 1.0 if risk_level == "LOW" else 0.5 if risk_level == "HIGH" else 0.75
        }
    else:
        state["risk_assessment"] = {"risk_level": "MODERATE", "position_size_multiplier": 0.75}
    
    return state

def portfolio_manager_node(state: TradingState) -> TradingState:
    """Agent 4: Makes final buy/sell/hold decision based on all analyses."""
    
    tech_signal = state.get("technical_signals", {}).get("gold", {}).get("signal", "NEUTRAL")
    tech_confidence = state.get("technical_signals", {}).get("gold", {}).get("confidence", 0.5)
    sentiment = state.get("sentiment_score", 0.5)
    risk = state.get("risk_assessment", {}).get("risk_level", "MODERATE")
    
    # Decision logic combining all inputs
    decision = "HOLD"
    confidence_final = 0.5
    reasoning = []
    
    # Technical analysis weighting
    if tech_signal == "BUY" and tech_confidence > 0.6:
        decision = "BUY"
        confidence_final = tech_confidence
        reasoning.append(f"Technical indicators show BUY signal (RSI: {state.get('technical_signals', {}).get('gold', {}).get('rsi', 'N/A')})")
    
    elif tech_signal == "SELL" and tech_confidence > 0.6:
        decision = "SELL"
        confidence_final = tech_confidence
        reasoning.append("Technical indicators show SELL signal")
    
    # Sentiment adjustment
    if sentiment > 0.6 and decision == "BUY":
        confidence_final = min(0.95, confidence_final * 1.2)
        reasoning.append(f"Positive sentiment ({sentiment:.2f}) reinforces the decision")
    elif sentiment > 0.6 and decision != "BUY":
        decision = "BUY"
        confidence_final = 0.55
        reasoning.append(f"Positive sentiment ({sentiment:.2f}) overrides neutral technicals")
    elif sentiment < 0.4 and decision == "BUY":
        decision = "HOLD"
        confidence_final = 0.4
        reasoning.append("Negative sentiment overrides technical signal")
    
    # Risk adjustment
    if risk == "HIGH" and decision == "BUY":
        decision = "HOLD"
        reasoning.append("HIGH risk levels suggest waiting for better entry")
    elif risk == "HIGH" and decision == "SELL":
        reasoning.append("HIGH risk levels support SELL decision")
    
    # Calculate position size based on risk
    position_size = 0
    if decision != "HOLD":
        base_size = 0.02  # 2% of portfolio per trade
        risk_multiplier = state.get("risk_assessment", {}).get("position_size_multiplier", 0.75)
        position_size = base_size * risk_multiplier * confidence_final
    
    final_decision = {
        "decision": decision,
        "confidence": confidence_final,
        "reasoning": reasoning,
        "position_size": position_size,
        "entry_horizon": "IMMEDIATE" if decision != "HOLD" else "NONE",
        "technical_signal_used": tech_signal,
        "sentiment_score": sentiment,
        "risk_level": risk,
        "timestamp": datetime.now().isoformat()
    }
    
    state["final_decision"] = final_decision
    state["timestamp"] = datetime.now().isoformat()
    return state

def build_trading_graph():
    """Build the LangGraph state machine."""
    workflow = StateGraph(TradingState)
    
    # Add nodes
    workflow.add_node("technical_analyst", technical_analyst_node)
    workflow.add_node("sentiment_analyst", sentiment_analyst_node)
    workflow.add_node("risk_manager", risk_manager_node)
    workflow.add_node("portfolio_manager", portfolio_manager_node)
    
    # Define the flow (parallel analysis, then sequential synthesis)
    workflow.set_entry_point("technical_analyst")
    workflow.add_edge("technical_analyst", "sentiment_analyst")
    workflow.add_edge("sentiment_analyst", "risk_manager")
    workflow.add_edge("risk_manager", "portfolio_manager")
    workflow.add_edge("portfolio_manager", END)
    
    # Compile with memory for state persistence
    memory = MemorySaver()
    return workflow.compile(checkpointer=memory)

c:\Users\dane\Desktop\Python Rag\Ragenv\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


ImportError: cannot import name 'MemorySaver' from 'langgraph.checkpoint' (unknown location)

# 3. Backtesting Engine
Before live deployment, thoroughly backtest your strategies

In [ ]:
# backtester.py
import pandas as pd
import numpy as np
from backtesting import Backtest, Strategy
from backtesting.lib import crossover
import yfinance as yf

class HybridGoldStrategy(Strategy):
    """
    Hybrid strategy for gold using RSI + MACD + sentiment filter.
    """
    
    rsi_period = 14
    rsi_oversold = 30
    rsi_overbought = 70
    
    def init(self):
        # Calculate indicators
        self.rsi = self.I(self.calculate_rsi, self.data.Close, self.rsi_period)
        
    def calculate_rsi(self, prices, period):
        """RSI calculation for backtesting."""
        deltas = np.diff(prices)
        seed = deltas[:period+1]
        up = seed[seed >= 0].sum() / period
        down = -seed[seed < 0].sum() / period
        rs = up / down
        rsi = np.zeros_like(prices)
        rsi[:period] = 100. - 100. / (1. + rs)
        
        for i in range(period, len(prices)):
            delta = deltas[i-1]
            if delta > 0:
                upval = delta
                downval = 0.
            else:
                upval = 0.
                downval = -delta
            
            up = (up * (period - 1) + upval) / period
            down = (down * (period - 1) + downval) / period
            rs = up / down if down != 0 else float('inf')
            rsi[i] = 100. - 100. / (1. + rs)
            
        return rsi
    
    def next(self):
        # Simple rule-based strategy
        if not self.position:
            # Buy signal: RSI oversold and price above 200-day SMA
            if self.rsi[-1] < self.rsi_oversold:
                self.buy(size=0.01, sl=self.data.Close[-1] * 0.98)
        else:
            # Sell signal: RSI overbought
            if self.rsi[-1] > self.rsi_overbought:
                self.position.close()

def run_backtest(symbol: str = "GC=F", start_date: str = "2020-01-01", end_date: str = "2024-12-31"):
    """Run backtest on gold futures data."""
    
    # Download historical data
    data = yf.download(symbol, start=start_date, end=end_date)
    
    if data.empty:
        print(f"No data found for {symbol}")
        return None
    
    # Initialize and run backtest
    bt = Backtest(data, HybridGoldStrategy, cash=10000, commission=.002)
    results = bt.run()
    
    # Print performance metrics
    print(f"\n=== Backtest Results for {symbol} ===")
    print(f"Start Date: {start_date}")
    print(f"End Date: {end_date}")
    print(f"Initial Balance: $10,000")
    print(f"Final Balance: ${results['Equity Final [$]']:.2f}")
    print(f"Total Return: {results['Return [%]']:.2f}%")
    print(f"Sharpe Ratio: {results['Sharpe Ratio']:.2f}")
    print(f"Max Drawdown: {results['Max. Drawdown [%]']:.2f}%")
    print(f"Win Rate: {results['Win Rate [%]']:.2f}%")
    print(f"Total Trades: {results['# Trades']}")
    
    return results

# 4. Risk Management System

In [ ]:
# risk_manager.py
from dataclasses import dataclass
from typing import Optional
import numpy as np

@dataclass
class RiskParameters:
    max_risk_per_trade: float = 0.02      # 2% of capital per trade
    max_daily_risk: float = 0.06           # 6% max loss per day
    atr_multiplier_stop: float = 1.8       # ATR-based stop-loss
    atr_multiplier_target: float = 4.5     # ATR-based take-profit
    max_position_size: float = 0.25        # Max 25% of portfolio in one position

class PositionSizer:
    """Calculate position sizes based on ATR and risk parameters."""
    
    def __init__(self, account_balance: float, risk_params: RiskParameters = None):
        self.account_balance = account_balance
        self.risk_params = risk_params or RiskParameters()
        
    def calculate_position_size(self, entry_price: float, atr: float, direction: str) -> dict:
        """
        Calculate position size, stop-loss, and take-profit levels.
        
        Args:
            entry_price: Proposed entry price
            atr: Current Average True Range
            direction: "BUY" or "SELL"
            
        Returns:
            Dictionary with position details
        """
        # Calculate stop-loss distance
        stop_distance = atr * self.risk_params.atr_multiplier_stop
        
        if direction == "BUY":
            stop_loss = entry_price - stop_distance
            take_profit = entry_price + (atr * self.risk_params.atr_multiplier_target)
        else:  # SELL
            stop_loss = entry_price + stop_distance
            take_profit = entry_price - (atr * self.risk_params.atr_multiplier_target)
        
        # Calculate position size based on risk
        risk_amount = self.account_balance * self.risk_params.max_risk_per_trade
        position_size = risk_amount / stop_distance
        
        # Apply constraints
        max_position_value = self.account_balance * self.risk_params.max_position_size
        position_value = position_size * entry_price
        
        if position_value > max_position_value:
            position_size = max_position_value / entry_price
        
        return {
            "position_size": position_size,
            "stop_loss": stop_loss,
            "take_profit": take_profit,
            "risk_amount_usd": risk_amount,
            "entry_price": entry_price
        }

class PortfolioTracker:
    """Track portfolio performance and monitor risk limits."""
    
    def __init__(self, initial_balance: float):
        self.initial_balance = initial_balance
        self.current_balance = initial_balance
        self.daily_pnl = 0
        self.trade_history = []
        self.daily_reset()
        
    def daily_reset(self):
        """Reset daily tracking."""
        self.daily_pnl = 0
        self.daily_start_balance = self.current_balance
        
    def check_daily_loss_limit(self, risk_params: RiskParameters) -> bool:
        """Check if daily loss limit has been exceeded."""
        daily_loss = self.daily_start_balance - self.current_balance if self.daily_start_balance else 0
        daily_loss_pct = daily_loss / self.daily_start_balance if self.daily_start_balance else 0
        
        if daily_loss_pct > risk_params.max_daily_risk:
            return False  # Stop trading for the day
        return True  # Continue trading
        
    def update_balance(self, new_balance: float):
        """Update current balance."""
        self.current_balance = new_balance

# 🚀 5. Main Execution & Paper Trading

In [ ]:
# main.py
import os
from dotenv import load_dotenv
from datetime import datetime
import json
import time

from data_fetchers import GoldSilverPriceFetcher, NewsFetcher
from trading_agent import build_trading_graph, TradingState
from risk_manager import RiskParameters, PositionSizer
from backtester import run_backtest

# Load environment variables
load_dotenv()

def initialize_paper_trading():
    """Initialize paper trading connection with Alpaca."""
    try:
        from alpaca.trading.client import TradingClient
        from alpaca.trading.requests import MarketOrderRequest
        from alpaca.trading.enums import OrderSide, TimeInForce
        
        trading_client = TradingClient(
            api_key=os.getenv("ALPACA_API_KEY"),
            secret_key=os.getenv("ALPACA_SECRET_KEY"),
            paper=True
        )
        return trading_client
    except ImportError:
        print("Alpaca package not installed. Running in simulation mode.")
        return None

def run_live_analysis():
    """Run a single analysis cycle and return trading signal."""
    
    print(f"\n{'='*60}")
    print(f"GOLD & SILVER TRADING AGENT - ANALYSIS CYCLE")
    print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'='*60}\n")
    
    # Fetch current prices
    price_fetcher = GoldSilverPriceFetcher()
    prices = price_fetcher.get_current_prices()
    
    print("📊 CURRENT PRICES:")
    for metal, data in prices.items():
        print(f"  {metal}: ${data.price:.2f} USD")
    
    # Initialize trading graph
    graph = build_trading_graph()
    
    # Prepare initial state
    initial_state: TradingState = {
        "messages": [],
        "gold_price": prices.get("GOLD", {}).price if "GOLD" in prices else 0,
        "silver_price": prices.get("SILVER", {}).price if "SILVER" in prices else 0,
        "technical_signals": {},
        "sentiment_score": 0.5,
        "sentiment_details": [],
        "risk_assessment": {},
        "final_decision": {},
        "timestamp": datetime.now().isoformat()
    }
    
    # Run the agent graph
    print("\n🤖 AGENT TEAM ANALYSIS IN PROGRESS...")
    result = graph.invoke(initial_state, config={"configurable": {"thread_id": "gold_silver_session"}})
    
    # Display results
    final = result.get("final_decision", {})
    print(f"\n📈 FINAL DECISION: {final.get('decision', 'HOLD')}")
    print(f"  Confidence: {final.get('confidence', 0.5):.1%}")
    
    if final.get('reasoning'):
        print("\n  Reasoning:")
        for reason in final.get('reasoning', []):
            print(f"    • {reason}")
    
    print(f"\n📊 DETAILED BREAKDOWN:")
    print(f"  Technical Signal: {final.get('technical_signal_used', 'NEUTRAL')}")
    print(f"  Sentiment Score: {final.get('sentiment_score', 0.5):.2f}")
    print(f"  Risk Level: {final.get('risk_level', 'MODERATE')}")
    
    if final.get('position_size', 0) > 0:
        print(f"  Recommended Position Size: {final.get('position_size', 0):.1%} of portfolio")
    
    return result

def run_scheduled_analysis(interval_minutes: int = 60):
    """Run analysis continuously at specified interval."""
    
    print(f"\n🚀 Starting Gold/Silver Trading Agent")
    print(f"Interval: Every {interval_minutes} minutes")
    print(f"Press Ctrl+C to stop\n")
    
    cycle_count = 0
    
    try:
        while True:
            cycle_count += 1
            print(f"\n{'🔄'*30}")
            print(f"CYCLE #{cycle_count}")
            
            result = run_live_analysis()
            
            # Save result to file for historical tracking
            with open(f"decision_log_{datetime.now().strftime('%Y%m%d')}.json", "a") as f:
                json.dump(result.get("final_decision", {}), f)
                f.write("\n")
            
            print(f"\n⏳ Next analysis in {interval_minutes} minutes...")
            time.sleep(interval_minutes * 60)
            
    except KeyboardInterrupt:
        print("\n\n👋 Shutting down gracefully...")

def main():
    """Main entry point."""
    
    print("""
    ╔══════════════════════════════════════════════════════╗
    ║     GOLD & SILVER AI TRADING AGENT - v1.0            ║
    ║     Multi-Agent System with LangGraph                ║
    ╚══════════════════════════════════════════════════════╝
    """)
    
    print("Select mode:")
    print("  1. Run single analysis")
    print("  2. Run scheduled analysis (every 60 minutes)")
    print("  3. Run backtest on historical data")
    print("  4. Exit")
    
    choice = input("\nEnter choice (1-4): ").strip()
    
    if choice == "1":
        run_live_analysis()
    elif choice == "2":
        interval = input("Interval in minutes (default 60): ").strip()
        interval_min = int(interval) if interval else 60
        run_scheduled_analysis(interval_min)
    elif choice == "3":
        symbol = input("Symbol (GC=F for gold, SI=F for silver, default GC=F): ").strip()
        if not symbol:
            symbol = "GC=F"
        start_date = input("Start date (YYYY-MM-DD, default 2020-01-01): ").strip()
        if not start_date:
            start_date = "2020-01-01"
        end_date = input("End date (YYYY-MM-DD, default 2024-12-31): ").strip()
        if not end_date:
            end_date = "2024-12-31"
        run_backtest(symbol, start_date, end_date)
    else:
        print("Exiting. Goodbye!")

if __name__ == "__main__":
    main()